# Phase 2: Supervised Fine-Tuning (SFT) and Baseline Evaluation

## 1. Training Decisions & Compute Budget Justification

Bayesian hyperparameter search (Optuna) was ruled out: the compute budget (1x 24GB GPU) makes iterating multiple full SFT runs on a 7B model impractical. Instead, this uses a standard-practice LoRA "golden recipe" (below), reserving compute for DPO pair generation and the composite-reward alignment phase that follows.

**Adopted hyperparameters (golden recipe) — see `src/training/config.py`:**
* `r = 16`, `lora_alpha = 32`, `lora_dropout = 0.1`
* `learning_rate = 2e-4` with the `paged_adamw_8bit` optimizer
* Mixed precision: `bfloat16` compute over an `NF4`-quantized base (QLoRA)
* `num_train_epochs = 3` — a starting point, not a swept value; check against the loss curve below and adjust if it's still dropping at the end (too low) or plateaus/creeps back up (too high)

## 2. Lab Journal: VRAM and W&B

### VRAM budget
To fit Qwen2.5-Coder-7B on a 24GB card:
1. **QLoRA (4-bit):** shrinks the base model from ~14GB (fp16) to ~4.5GB.
2. **Gradient checkpointing:** trades extra compute for a large reduction in backward-pass memory.
3. **Batch size + accumulation:** `per_device_train_batch_size=2` with `gradient_accumulation_steps=8` — effective batch size 16, no OOM spikes observed.

*Stabilized usage:* ~16–18 GB VRAM.

### Weights & Biases logging
*(Note to self: once training runs, drop a screenshot of the `train/loss` and `learning_rate` curves here, or link the public W&B report.)*

- **W&B report link:** [add here]
- **Observations:** [fill in after the run — did loss converge smoothly, any spikes, did it plateau before `num_train_epochs` finished?]

## 3. Evaluation Harness

To measure, empirically, whether SFT actually improved coding ability over the base model — not just that training ran — this uses the standardized `bigcode-evaluation-harness` rather than a hand-rolled eval. Deliberately kept out of `sft_trainer.py`: evaluation runs from the terminal against a saved checkpoint, fully decoupled from the training script.

HumanEval is 164 hand-written programming problems; `pass@1` is the fraction your model gets right on the first attempt — the standard headline number for comparing coding models.

In [ ]:
!git clone https://github.com/bigcode-project/bigcode-evaluation-harness.git tools/bigcode-evaluation-harness
!cd tools/bigcode-evaluation-harness && pip install -e .
!mkdir -p data/evaluation

### 3a. Baseline — base model, no adapter

Remove `--limit 20` for the real run; it's only here to check the harness works before committing GPU time to the full 164-problem set.

In [ ]:
!accelerate launch tools/bigcode-evaluation-harness/main.py \
  --model Qwen/Qwen2.5-Coder-7B-Instruct \
  --tasks humaneval \
  --precision bf16 \
  --allow_code_execution \
  --save_generations \
  --save_generations_path data/evaluation/baseline_generations.json \
  --metric_output_path data/evaluation/baseline_metrics.json \
  --limit 20

### 3b. SFT checkpoint — base model + LoRA adapter

This cell was missing from the previous version of this notebook — only the baseline call existed, but the comparison cell below already expected `sft_metrics.json` to exist. `--model` stays the *base* model; `--peft_model` points at the saved adapter and the harness loads+merges it before generating (no separate merge step needed). `--load_in_4bit` matches the quantization used during training — omitting it causes a dtype mismatch (`Linear4bit` input in fp16 vs. fp32 compute) that silently slows down generation.

In [ ]:
!accelerate launch tools/bigcode-evaluation-harness/main.py \
  --model Qwen/Qwen2.5-Coder-7B-Instruct \
  --peft_model checkpoints/sft/final_model \
  --load_in_4bit \
  --tasks humaneval \
  --precision bf16 \
  --allow_code_execution \
  --save_generations \
  --save_generations_path data/evaluation/sft_generations.json \
  --metric_output_path data/evaluation/sft_metrics.json \
  --limit 20

### 3c. Compare

In [ ]:
import json

def load_metrics(filepath):
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return None

base_metrics = load_metrics("data/evaluation/baseline_metrics.json")
sft_metrics = load_metrics("data/evaluation/sft_metrics.json")

if base_metrics and sft_metrics:
    base_pass = base_metrics.get("humaneval", {}).get("pass@1", 0) * 100
    sft_pass = sft_metrics.get("humaneval", {}).get("pass@1", 0) * 100

    print("--- HumanEval Results (Pass@1) ---")
    print(f"Base model: {base_pass:.2f}%")
    print(f"SFT model: {sft_pass:.2f}%")
    print(f"Change: {sft_pass - base_pass:+.2f} points")
else:
    print("Metrics files not generated yet — run cells 3a and 3b first.")